In [1]:
import os
from dotenv import load_dotenv
import gradio as gr
import google.generativeai as genai

In [2]:
load_dotenv(override = True)
api_key = os.getenv("GEM_API_KEY")
genai.configure(api_key = api_key)

In [3]:
model = genai.GenerativeModel("gemini-2.5-flash-lite")

In [4]:
system_message = "You are a helpful assistant in a clothes store. You should try to gently encourage \
the customer to try items that are on sale. Hats are 60% off, and most other items are 50% off. \
For example, if the customer says 'I'm looking to buy a hat', \
you could reply something like, 'Wonderful - we have lots of hats - including several that are part of our sales event.'\
Encourage the customer to buy hats if they are unsure what to get."

In [5]:
def chat(message, history):
    """
    Handles a single turn of conversation for a clothes store assistant chatbot using the Gemini model.
    Streams the AI's response back incrementally for real-time display in Gradio.

    The assistant encourages customers to buy sale items, especially hats (60% off),
    and adapts responses if the user mentions items the store does not sell (e.g., belts).

    Args:
        message (str):
            The latest user message to be processed.
        history (list[dict]):
            A list of previous conversation turns in Gradio's format:
            Each dict should have:
                - "role": either "user" or "assistant"
                - "content": the text of that message.

    Yields:
        str:
            Partial and progressively longer chunks of the AI's reply,
            enabling real-time streaming in the UI.

    Notes:
        - Uses `system_message` as the base system prompt and adapts it if needed.
        - Converts Gradio's message history to the Gemini API's expected format before calling.
    """
    relevant_system_message = system_message
    if "belt" in message.lower():
        relevant_system_message += " The store does not sell belts; suggest other sale items."

    convo_history = [{"role": "user", "parts": [relevant_system_message]}]

    for msg in history:
        if msg["role"] == "user":
            convo_history.append({"role": "user", "parts": [msg["content"]]})
        elif msg["role"] == "assistant":
            convo_history.append({"role": "model", "parts": [msg["content"]]})

  
    convo_history.append({"role": "user", "parts": [message]})

    stream = model.generate_content(convo_history, stream=True)
    response = ""
    for chunk in stream:
        if chunk.text:
            response += chunk.text
            yield response

In [6]:
gr.ChatInterface(fn=chat, type="messages").launch()

* Running on local URL:  http://127.0.0.1:7860
* To create a public link, set `share=True` in `launch()`.
